# Part I: Data Acquisition & Sourcing

> **Objective:** Ingest raw market data, macroeconomic indicators, and historical event logs to construct clean, unified panels for shock analysis.

### Workflow
1. **Directory Setup:** Establish local environments for `/data/raw/` and `/data/processed/`.
2. **Ingestion:** Load BIST 100 historicals, macroeconomic context, and the events database.
3. **Alignment:** Standardize datetime indices and forward-fill missing market days.
4. **Aggregation:** Compile the final datasets for downstream modeling and exploratory data analysis.

### Data Dictionary

**Raw Sources (Inputs)**
* `bist100_raw.csv` — Daily price and volume data (Primary target variable)
* `macro_indicators_raw.csv` — Contextual economic metrics (Control variables)
* `events_raw.csv` — Database of isolated shocks (Event triggers)

**Generated Datasets (Outputs)**
* `unified_bist_data.csv` — Merged time-series with event flags (Exploratory Data Analysis)
* `unified_bist_enriched.csv` - Multivariate dataframe with volatility and trend features (for ML/Modeling).
* `shock_analysis_summary.csv` — Summarized event-window panel (Hypothesis Testing)

## [1] BIST Data Collection and Analysis

Loads all required data science libraries and establishes the project directory structure. Specifically, it locates the `raw` and `processed` data folders, and will automatically create these directories if they do not already exist in the environment.

In [1]:
import os
import warnings
from pathlib import Path
from datetime import datetime
from evds import evdsAPI
import yfinance as yf

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Suppress warnings and set default plot style
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams['figure.figsize'] = (12, 6)

# Set up project paths based on the exact folder structure
# NOTEBOOK_DIR is data_collection/jupyter_notebook/
NOTEBOOK_DIR = Path.cwd()

# Step up one level to data_collection/
PROJECT_ROOT = NOTEBOOK_DIR.parent 

# Point directly to the existing sibling folders
RAW_DIR = PROJECT_ROOT / "raw"
PROCESSED_DIR = PROJECT_ROOT / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {RAW_DIR}")
print(f"Processed data: {PROCESSED_DIR}")

Project root: /Users/yigitkeskin/Desktop/DSA_210_Project/data_collection
Raw data: /Users/yigitkeskin/Desktop/DSA_210_Project/data_collection/raw
Processed data: /Users/yigitkeskin/Desktop/DSA_210_Project/data_collection/processed


## [2] Environment Setup: EVDS Installation
Ensures the `evds` library is installed and updated to the latest version to allow programmatic access to the TCMB (Central Bank) Electronic Data Delivery System.

In [2]:
%pip install evds --upgrade -q

Note: you may need to restart the kernel to use updated packages.


## [3] Data Ingestion: BIST 100 Index
Retrieves historical daily price data for the BIST 100 index (XU100.IS) from Yahoo Finance and stores it in the raw data directory.

In [3]:
# Define the target file path using the RAW_DIR established in Cell 1
bist_raw_path = RAW_DIR / "bist100_raw.csv"

print("Downloading BIST 100 data from Yahoo Finance...")

# Download daily data from 2015 to the current period
bist_df = yf.download("XU100.IS", start="2015-01-01", end="2026-04-05", progress=False)

# Reset index to convert 'Date' from index to a column
bist_df = bist_df.reset_index()

# Standardize column names: lowercase and handle yfinance multi-index tuples
bist_df.columns = [c[0].lower() if isinstance(c, tuple) else c.lower() for c in bist_df.columns]

# Export to CSV
bist_df.to_csv(bist_raw_path, index=False)

print(f"Saved BIST 100 data to {bist_raw_path}")
print(f"Final Data Shape: {bist_df.shape}")

Saved BIST 100 data to /Users/yigitkeskin/Desktop/DSA_210_Project/data_collection/raw/bist100_raw.csv
Final Data Shape: (2817, 6)


## [4] Data Ingestion: Macroeconomic Indicators
Retrieves the Consumer Price Index (CPI) and USD/TRY Exchange Rate from the Central Bank of the Republic of Türkiye (CBRT) EVDS system.

In [4]:
# Load API Key from secrets.txt
# This file should be in your 'jupyter_notebook' folder alongside this script
with open("secrets.txt", "r") as f:
    MY_API_KEY = f.read().strip()

# Initialize the client
evds_client = evdsAPI(MY_API_KEY)

# TP.FG.J0: Consumer Price Index (Monthly)
# TP.DK.USD.A.YTL: USD/TRY Exchange Rate (Daily)
series_codes = ['TP.FG.J0', 'TP.DK.USD.A.YTL']

print("Downloading Macroeconomic data from TCMB EVDS...")

# Fetch data from Jan 2015 to April 2026
macro_df = evds_client.get_data(
    series_codes, 
    startdate="01-01-2015", 
    enddate="05-04-2026"
)

# Rename columns for clarity
macro_df.columns = ['date', 'cpi', 'usd_try']

# Save to the existing raw folder using the Path variable from Cell 1
macro_raw_path = RAW_DIR / "macro_indicators_raw.csv"
macro_df.to_csv(macro_raw_path, index=False)

print(f"Success: Saved Macro data to {macro_raw_path}")
print(f"Dataset Shape: {macro_df.shape}")

Success: Saved Macro data to /Users/yigitkeskin/Desktop/DSA_210_Project/data_collection/raw/macro_indicators_raw.csv
Dataset Shape: (136, 3)


## [5] Data Preparation: Significant Market Shocks (2015-2026)
Constructs a master timeline of geopolitical, domestic political, and economic events that potentially impacted the BIST index, saving the dataset to the raw directory.

In [5]:
# Use the RAW_DIR path established in Cell 1
events_raw_path = RAW_DIR / "events_raw.csv"

# The Complete Master List of Shocks
# Categorized by type and severity for impact analysis
data = [
    # --- 2015-2017: EARLY PERIOD SHOCKS ---
    ["2015-06-08", "Post-Election Market Uncertainty", "Political/Economic", "High"],
    ["2015-11-24", "Russia Jet Downed", "Geopolitical", "High"],
    ["2016-07-15", "Coup Attempt", "Domestic Political", "Extreme"],
    ["2017-04-16", "Constitutional Referendum", "Domestic Political", "Medium"],
    ["2016-05-05", "PM Davutoglu Resignation/Removal", "Domestic Political", "High"],

    # --- 2018-2021: CURRENCY & POLICY SHOCKS ---
    ["2018-08-10", "US-Turkey Currency Crisis", "Monetary Policy", "Extreme"],
    ["2019-03-31", "Local Elections", "Domestic Political", "Medium"],
    ["2020-03-11", "COVID-19 Pandemic Declaration", "Global Health", "Extreme"],
    ["2021-03-12", "Economic Reform Package Unveiled", "Structural Reform", "Medium"],
    ["2021-03-20", "CBRT Governor Dismissal (Agbal)", "Monetary Policy", "Extreme"],
    ["2021-12-20", "KKM Scheme Announcement", "Monetary Policy", "High"],
    
    # --- 2022-2024: WAR & PIVOT PERIOD ---
    ["2022-02-24", "Russia-Ukraine War Starts", "Global Geopolitical", "High"],
    ["2022-12-14", "Ekrem Imamoglu Sentencing/Political Ban", "Domestic Political", "Medium"],
    ["2023-02-06", "Kahramanmaras Earthquake", "Natural Disaster", "Extreme"],
    ["2023-05-14", "General Elections Round 1", "Domestic Political", "High"],
    ["2023-05-28", "General Elections Round 2", "Domestic Political", "High"],
    ["2023-06-22", "Pivot to Orthodox Policy", "Monetary Policy", "Medium"],
    ["2023-10-07", "Gaza Conflict Begins", "Global Geopolitical", "Medium"],
    
    # --- 2025-2026: RECENT VOLATILITY ---
    ["2025-01-15", "Major Interest Rate Cut (500bps)", "Monetary Policy", "High"],
    ["2025-03-20", "Emergency Overnight Rate Hike (46%)", "Monetary Policy", "Extreme"],
    ["2026-01-05", "Regional Conflict Escalation", "Geopolitical", "High"],
    ["2026-01-22", "CBRT Rate Cut to 37%", "Monetary Policy", "Medium"],
    ["2026-02-03", "Inflation Hits 4-Year Low (30.65%)", "Economic Indicator", "High"],
    ["2026-03-03", "Unexpected Inflation Spike (31.53%)", "Economic Indicator", "Medium"]
]

# Create and Sort the DataFrame
events_df = pd.DataFrame(data, columns=["date", "event_name", "category", "severity"])
events_df['date'] = pd.to_datetime(events_df['date'])
events_df = events_df.sort_values(by='date')

# Save to the raw folder
events_df.to_csv(events_raw_path, index=False)

print(f"Success: Master Events Dataset created with {len(events_df)} significant shocks.")
print(f"File saved to: {events_raw_path}")

# Display the timeline
print("\n--- FULL SHOCK TIMELINE (2015-2026) ---")
display(events_df)

Success: Master Events Dataset created with 24 significant shocks.
File saved to: /Users/yigitkeskin/Desktop/DSA_210_Project/data_collection/raw/events_raw.csv

--- FULL SHOCK TIMELINE (2015-2026) ---


,date,event_name,category,severity
0,2015-06-08,Post-Election Market Uncertainty,Political/Economic,High
1,2015-11-24,Russia Jet Downed,Geopolitical,High
4,2016-05-05,PM Davutoglu Resignation/Removal,Domestic Political,High
2,2016-07-15,Coup Attempt,Domestic Political,Extreme
3,2017-04-16,Constitutional Referendum,Domestic Political,Medium
5,2018-08-10,US-Turkey Currency Crisis,Monetary Policy,Extreme
6,2019-03-31,Local Elections,Domestic Political,Medium
7,2020-03-11,COVID-19 Pandemic Declaration,Global Health,Extreme
8,2021-03-12,Economic Reform Package Unveiled,Structural Reform,Medium
9,2021-03-20,CBRT Governor Dismissal (Agbal),Monetary Policy,Extreme


## [6] Data Preprocessing and Integration
Merges the BIST price data, macroeconomic indicators, and the shock timeline into a single unified dataset. This process includes forward-filling monthly data and calculating daily percentage returns.

In [6]:
# 1. Load Raw Data using the variables from Cell 1
try:
    bist = pd.read_csv(RAW_DIR / "bist100_raw.csv", parse_dates=['date'])
    macro = pd.read_csv(RAW_DIR / "macro_indicators_raw.csv", parse_dates=['date'])
    events = pd.read_csv(RAW_DIR / "events_raw.csv", parse_dates=['date'])
    print("All raw files loaded successfully.")
except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    print("Check if the filenames in your 'raw' folder match the code exactly.")

# --- STEP A: Create the Unified Daily File ---
# Left merge ensures we keep every trading day from the BIST dataset
df_daily = pd.merge(bist, macro, on='date', how='left')

# Sort by date for proper time-series operations
df_daily = df_daily.sort_values('date')

# Forward-fill monthly CPI and daily Exchange Rates to fill missing trading day values
df_daily[['cpi', 'usd_try']] = df_daily[['cpi', 'usd_try']].ffill()

# Flag dates that match our specific shock events
df_daily['is_shock'] = df_daily['date'].isin(events['date']).astype(int)

# Calculate Daily Returns for analysis
df_daily['daily_return_pct'] = df_daily['close'].pct_change() * 100

# 2. Save Daily Processed to the established processed directory
output_path = PROCESSED_DIR / "unified_bist_data.csv"
df_daily.to_csv(output_path, index=False)

print(f"Created unified_bist_data.csv in {PROCESSED_DIR}")
print(f"Final Dataset Shape: {df_daily.shape}")

# Preview the results
display(df_daily.head())

All raw files loaded successfully.
Created unified_bist_data.csv in /Users/yigitkeskin/Desktop/DSA_210_Project/data_collection/processed
Final Dataset Shape: (2817, 10)


,date,close,high,low,open,volume,cpi,usd_try,is_shock,daily_return_pct
0,2015-01-02,854.584473,857.716475,849.404507,854.474488,32803700000,NaN,NaN,0,NaN
1,2015-01-05,864.621399,867.953413,854.109492,854.948477,64458800000,NaN,NaN,0,1.174480
2,2015-01-06,869.091370,869.113342,855.742442,866.504347,69832100000,NaN,NaN,0,0.516986
3,2015-01-07,867.761414,876.255387,864.973464,869.246391,79194900000,NaN,NaN,0,-0.153028
4,2015-01-08,876.891357,879.186328,872.599388,874.044388,73176200000,NaN,NaN,0,1.052126


## [7] Event Impact Analysis: Window Calculations
Calculates the average daily returns for a 7-day window before and after each identified shock. This allows for a direct comparison (impact delta) to quantify how specific events shifted market volatility.

In [7]:
# --- STEP B: Create the Shock Analysis Summary ---
shock_summary = []

# Using the events dataframe and df_daily created in the previous steps
for index, row in events.iterrows():
    event_date = row['date']
    
    # Define a 7-day window (1 week) before and after the shock
    # This window typically captures 5 full trading days
    mask_pre = (df_daily['date'] < event_date) & (df_daily['date'] >= event_date - pd.Timedelta(days=7))
    mask_post = (df_daily['date'] > event_date) & (df_daily['date'] <= event_date + pd.Timedelta(days=7))
    
    # Calculate the average daily return in these windows
    pre_return = df_daily.loc[mask_pre, 'daily_return_pct'].mean()
    post_return = df_daily.loc[mask_post, 'daily_return_pct'].mean()
    
    shock_summary.append({
        'event_name': row['event_name'],
        'date': event_date,
        'category': row['category'],
        'severity': row['severity'],
        'avg_return_before': pre_return,
        'avg_return_after': post_return,
        'impact_delta': post_return - pre_return if (pd.notnull(post_return) and pd.notnull(pre_return)) else None
    })

# Create the summary DataFrame
df_shocks = pd.DataFrame(shock_summary)

# Save to the processed folder using the path from Cell 1
summary_output_path = PROCESSED_DIR / "shock_analysis_summary.csv"
df_shocks.to_csv(summary_output_path, index=False)

print(f"✅ Created shock_analysis_summary.csv in {PROCESSED_DIR}")
print(f"Total events analyzed: {len(df_shocks)}")

# Preview the most impactful shocks
display(df_shocks.sort_values(by='impact_delta', ascending=True).head())

✅ Created shock_analysis_summary.csv in /Users/yigitkeskin/Desktop/DSA_210_Project/data_collection/processed
Total events analyzed: 24


,event_name,date,category,severity,avg_return_before,avg_return_after,impact_delta
3,Coup Attempt,2016-07-15,Domestic Political,Extreme,1.055161,-2.797475,-3.852636
13,Kahramanmaras Earthquake,2023-02-06,Natural Disaster,Extreme,-0.697968,-4.309158,-3.611190
14,General Elections Round 1,2023-05-14,Domestic Political,High,1.792477,-1.509078,-3.301555
10,KKM Scheme Announcement,2021-12-20,Monetary Policy,High,0.598012,-1.500750,-2.098762
5,US-Turkey Currency Crisis,2018-08-10,Monetary Policy,Extreme,0.559884,-1.318218,-1.878101


## [8] Feature Engineering: Market Sentiment & Stability Indicators
This section generates technical indicators to quantify the BIST 100's reaction to external shocks beyond simple price changes. We engineer:
* **7-Day Rolling Volatility**: Measures market "nervousness" and panic levels.
* **20-Day Moving Average (MA)**: Establishes the underlying price trend prior to a shock.
* **Deviation from Trend**: Quantifies the "snap" or percentage distance from the trend, identifying oversold/overbought conditions caused by specific events.

In [8]:
# 1. Rolling Volatility (Standard Deviation of returns)
# Helps identify if a 'shock' increased market nervousness
df_daily['volatility_7d'] = df_daily['daily_return_pct'].rolling(window=7).std()

# 2. Price Momentum (Moving Average)
# Identifies if the market was already in a trend before a shock hit
df_daily['ma_20d'] = df_daily['close'].rolling(window=20).mean()

# 3. Deviation from Trend
# Measures how far the price 'snapped' away from its average during an event
df_daily['dev_from_ma'] = (df_daily['close'] - df_daily['ma_20d']) / df_daily['ma_20d']

# 4. Save the enriched dataset to processed folder
enriched_output_path = PROCESSED_DIR / "unified_bist_enriched.csv"
df_daily.to_csv(enriched_output_path, index=False)

print(f"Feature engineering complete. Added volatility and momentum indicators.")
print(f"Enriched file saved to: {enriched_output_path}")

# Preview the new features
display(df_daily[['date', 'close', 'daily_return_pct', 'volatility_7d', 'dev_from_ma']].tail(10))

Feature engineering complete. Added volatility and momentum indicators.
Enriched file saved to: /Users/yigitkeskin/Desktop/DSA_210_Project/data_collection/processed/unified_bist_enriched.csv


,date,close,daily_return_pct,volatility_7d,dev_from_ma
2807,2026-03-23,13168.200195,0.923534,1.250711,-0.008328
2808,2026-03-24,12930.200195,-1.807384,1.368097,-0.022085
2809,2026-03-25,12963.900391,0.260632,1.295103,-0.015490
2810,2026-03-26,12727.099609,-1.826617,1.415320,-0.029483
2811,2026-03-27,12698.200195,-0.227070,1.016013,-0.027309
2812,2026-03-30,12626.400391,-0.565433,1.011909,-0.028749
2813,2026-03-31,12791.000000,1.303615,1.228668,-0.013982
2814,2026-04-01,12937.900391,1.148467,1.267621,-0.002675
2815,2026-04-02,13051.700195,0.879585,1.112991,0.005677
2816,2026-04-03,12936.400391,-0.883408,1.174457,-0.002660
